# C7-cnn-transfer — Practice p20 — Solution

For the stem, `conv1` updates $(r,J)$ from $(1,1)$ to $(7,2)$, then
`maxpool` updates it to $(11,4)$. Each of `layer1`'s three 3×3
convolutions adds $(3-1)J=8$ pixels; its 1×1 convolutions add zero
because $K-1=0$, yielding $11+3\cdot8=35$.

For a stride-one stack, a 3×3 layer adds two pixels and a 5×5 layer
adds four. Reaching at least 61 from $r_0=1$ therefore needs 30 or 15
layers respectively; their per-channel-pair costs are 270 and 375,
so the 3×3 design is cheaper.


In [ ]:
rf_stem = 11
rf_layer1 = 35
L3 = 30
L5 = 15
cost3 = 9 * 30
cost5 = 25 * 15
cheaper = "3x3"


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (no pretrained weights here)
SEED = 20260804
torch.manual_seed(SEED)

def rf(kernels, strides):
    r, jump = 1, 1
    for kernel, stride in zip(kernels, strides):
        r += (kernel - 1) * jump
        jump *= stride
    return r

assert rf([7, 3], [2, 2]) == rf_stem
assert rf([3] * L3, [1] * L3) >= 61
assert rf([3] * (L3 - 1), [1] * (L3 - 1)) < 61
assert rf([5] * L5, [1] * L5) >= 61
assert rf([5] * (L5 - 1), [1] * (L5 - 1)) < 61
rf_checks_ok = True

def ones_conv(K, stride):
    layer = nn.Conv1d(1, 1, K, stride=stride, bias=False)
    layer.weight = nn.Parameter(torch.ones(1, 1, K), requires_grad=False)
    return layer

mixed = nn.Sequential(ones_conv(3, 2), ones_conv(3, 1))
base = torch.zeros(1, 1, 41)
with torch.inference_mode():
    baseline = mixed(base)
center = baseline.shape[-1] // 2
rf_mixed_measured = 0
with torch.inference_mode():
    for i in range(base.shape[-1]):
        probe = base.clone()
        probe[0, 0, i] = 1.0
        rf_mixed_measured += int(mixed(probe)[0, 0, center] != baseline[0, 0, center])
rf_mixed_formula = rf([3, 3], [2, 1])
mixed_gap = abs(rf_mixed_formula - rf_mixed_measured)


### Answer check

In [ ]:
assert (rf_stem, rf_layer1) == (11, 35)
assert (L3, L5, cost3, cost5, cheaper) == (30, 15, 270, 375, "3x3")
assert rf_checks_ok
assert rf_mixed_formula == rf_mixed_measured == 7
assert mixed_gap == 0
